 Introduction

This notebook creates, applies and tests a system for Retrieval-Augmented 
Cybersecurity guideline generation system- generation (RAG). The system 
solves a practical problem for security experts: how to get at: 
Rapid, authoritative and fact-based advice on adversary methods and countermeasures.

Effective writing of Knowledge bases at MITRE will be covered by the scope. 
ATT&CK Enterprise Framework, dense retrieval using semantic embeddings, 
Generating guidelines from LLM: open source vs. closed source, reference-free evaluation. 
using RAGAS-style metrics.

Objectives:
Create a extensive cybersecurity knowledge base with more than 2000 samples. 
   from authoritative sources
 Use a dense retrieval method to obtain similar sentences to the query
 Implement an instruction-tuned LLM in a quantized manner to create practitioner guidelines
 Assess the pipeline by means of Context Relevance, Answer Relevance, 
   and Faithfulness metrics
 Critically consider trustworthiness, bias, ethics and governance.

  
Knowledge Base: MITRE ATT&CK Enterprise (Strumia et al., 2018)  
Embedding Model: all-MiniLM-L6-v2 (Reimers & Gurevych, 2019)  
Model size: 7B parameters  
Assessment: RAGAS-like metrics (Es et al., 2023)


Literature Review

Retrieval-Augmented Generation: Lewis et al. (2020) proposed RAG, 
a combination of dense retriever and sequence-to-sequence generator for grounding 
The LLM is tasked with generating LLM outputs in external knowledge and to reduce hallucination in knowledge-intensive tasks. 
tasks. This work is a direct inspiration for the architecture that is used in this assignment.

Karpukhin et al. (2020) showed that it was possible to achieve Dense Passage Retrieval with 
Using BERT-based embeddings, bi-encoder dense retrieval outperforms dense retrieval which outperforms even the most advanced dense retrieval methods like SimCaps and ALBERT by over 10%. 
outperforms open domain question answering with sparse BM25 retrieval. 
This is why it is perfectly acceptable to use all-MiniLM-L6-v2 over TF-IDF in this pipeline.

Sentence-BERT (Reimers and Gurevych, 2019) is an approach that builds sentence embeddings. 
Creating fixed-size sentence representations that are semantically meaningful using siamese networks 
network fine-tuning. The model used here is an all-MiniLM-L6-v2 model that is distilled. 
Variants that are able to show good MTEB benchmark performance while reducing computational cost.

Efficient similarity search in high-dimensional, dense vector spaces.efficient similarity search in high-dimensional, dense vector spaces (Johnson et al., 2021). 
Search on dense vectors at billions scale. At 2578 vectors, IndexFlatIP 
Allows for precise cosine query in almost no time.

Of the four quantization methods, NF4 was shown to be the most effective by Dettmers et al. (2023). 
With quantization, memory can be reduced by as much as 75%, however, the quality of the model is not compromised. 
Allowing the 7B parameter models to be run on consumer GPUs. This directly 
Justifies the BitsAndBytesConfig used to load Zephyr-7B on the T4.

Zephyr-7B: Tunstall et al. (2023) released Zephyr-7B-beta, and 
It was fine-tuned with an approach called Direct Preference Optimisation (DPO) from Mistral-7B. DPO 
Improves instruction without reinforcement learning, making 


RAGAS Evaluation: Es et al. (2023) suggested a reference-free method called RAGAS evaluation. 
framework to assess RAG systems based on Context Relevance, Answer Relevance, 
and Faithfulness with no human annotations.

MITRE ATT&CK: The ATT&CK framework was published by MITRE (Stram, et al., 2018). 
a formal Adversarial Knowledge Base of adversarial techniques being used by governments. 
the industry's definitive guide to security worldwide, it was the perfect authority for you to 
A cybersecurity RAG knowledge base.


In [1]:
# =============================================================================
# Section 1 : Environment Setup
# =============================================================================

import subprocess, sys

def upgrade(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", package])

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# Upgrade the three coupled packages to a consistent modern set
# transformers>=4.43 required for EncoderDecoderCache (peft dependency)
upgrade("transformers")
upgrade("accelerate")   # peft requires >=0.33.0 for clear_device_cache
upgrade("peft")  # ensure peft matches the upgraded transformers


# sentence-transformers: dense retrieval embeddings (Reimers & Gurevych, 2019)
upgrade("sentence-transformers")

upgrade("bitsandbytes")

# faiss-cpu: ANN vector search (Johnson et al., 2021)
install("faiss-cpu==1.8.0")

# langchain: RAG chain abstractions
install("langchain==0.2.5")
install("langchain-community==0.2.5")
install("langchain-huggingface==0.0.3")

# datasets: HuggingFace data loading
install("datasets==2.19.2")

# ragas: reference-free RAG evaluation (Es et al., 2023 arXiv:2309.15217)
install("ragas==0.1.9")

# Utilities
install("rouge-score==0.1.2")
install("nltk==3.8.1")
install("tiktoken==0.7.0")


print("Installation complete.")

Installation complete.


In [2]:
# =============================================================================
# : Seed and directory setup
# =============================================================================

import sys, os, random
import torch
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache"
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.makedirs("/kaggle/working/outputs", exist_ok=True)

print("Seed:", SEED)
print("Cache:", os.environ["HF_HOME"])
print("Outputs: /kaggle/working/outputs")
print("Setup complete.")

Seed: 42
Cache: /kaggle/working/hf_cache
Outputs: /kaggle/working/outputs
Setup complete.


In [3]:
# =============================================================================
#  Verification
# =============================================================================

import transformers, accelerate, sentence_transformers
import faiss, langchain, ragas
import pandas as pd

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("faiss:", faiss.__version__)
print("langchain:", langchain.__version__)
print("ragas:", ragas.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}, VRAM: {gpu.total_memory/(1024**3):.1f} GB")
else:
    print("WARNING: no GPU detected")

transformers: 5.9.0
accelerate: 1.13.0
sentence-transformers: 5.5.1
faiss: 1.8.0
langchain: 0.2.5
ragas: 0.1.9
pandas: 2.2.2
numpy: 1.26.4
GPU: Tesla T4, VRAM: 14.6 GB


In [5]:
# =============================================================================
# Section 2: Dataset Construction and Knowledge Base Assembly
#
#   Tier 1 - MITRE ATT&CK Enterprise JSON
# Target: 2000+ total samples saved to JSONL for downstream pipeline
# =============================================================================

import json
import os
import re
import time
import random
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_PATH = "/kaggle/working/outputs/cybersecurity_kb.jsonl"
STATS = {}  # track sample counts per source tier

In [6]:
# =============================================================================
# Tier 1: MITRE ATT&CK Enterprise Dataset
# Source: https://github.com/mitre/cti (STIX 2.0 format, CC BY 4.0 license)
# =============================================================================

MITRE_URL = (
    "https://raw.githubusercontent.com/mitre/cti/master/"
    "enterprise-attack/enterprise-attack.json"
)

print("Downloading MITRE ATT&CK Enterprise dataset...")
resp = requests.get(MITRE_URL, timeout=60)
resp.raise_for_status()
mitre_data = resp.json()
print(f"MITRE objects loaded: {len(mitre_data['objects'])}")

MITRE objects loaded: 25842


In [7]:
# =============================================================================
# Parse MITRE STIX objects into technique records
# STIX 2.0 schema: attack-pattern = technique, course-of-action = mitigation
# =============================================================================

def clean_text(text):
    # strip citation markers like (Citation: ...) and excess whitespace
    text = re.sub(r'\(Citation:[^)]+\)', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# index objects by id for relationship resolution
obj_index = {o['id']: o for o in mitre_data['objects'] if 'id' in o}

# extract mitigation relationships: technique_id -> list of mitigation descriptions
tech_to_mitigations = {}
for obj in mitre_data['objects']:
    if obj.get('type') == 'relationship' and obj.get('relationship_type') == 'mitigates':
        tgt = obj.get('target_ref', '')
        src = obj.get('source_ref', '')
        if tgt not in tech_to_mitigations:
            tech_to_mitigations[tgt] = []
        if src in obj_index:
            mit_obj = obj_index[src]
            mit_desc = clean_text(mit_obj.get('description', ''))
            mit_name = mit_obj.get('name', '')
            if mit_desc:
                tech_to_mitigations[tgt].append(f"{mit_name}: {mit_desc}")

techniques = []
for obj in mitre_data['objects']:
    if obj.get('type') != 'attack-pattern':
        continue
    if obj.get('x_mitre_deprecated', False) or obj.get('revoked', False):
        continue  # skip deprecated techniques to maintain data quality

    name = obj.get('name', '')
    desc = clean_text(obj.get('description', ''))
    detection = clean_text(obj.get('x_mitre_detection', ''))
    platforms = obj.get('x_mitre_platforms', [])
    tactics = [
        kc['phase_name'].replace('-', ' ').title()
        for kc in obj.get('kill_chain_phases', [])
        if kc.get('kill_chain_name') == 'mitre-attack'
    ]
    mitre_id = next(
        (r['external_id'] for r in obj.get('external_references', [])
         if r.get('source_name') == 'mitre-attack'), ''
    )
    mitigations = tech_to_mitigations.get(obj['id'], [])

    if name and desc:
        techniques.append({
            'id': mitre_id,
            'name': name,
            'description': desc,
            'detection': detection,
            'platforms': platforms,
            'tactics': tactics,
            'mitigations': mitigations
        })

print(f"Valid MITRE techniques parsed: {len(techniques)}")

Valid MITRE techniques parsed: 697


In [8]:
# =============================================================================
# Convert each MITRE technique into multiple Q&A records
# Each technique yields up to 4 distinct Q&A pairs covering different angles
# =============================================================================

def make_mitre_qa(tech):
    pairs = []
    name = tech['name']
    mid  = tech['id']
    src  = f"MITRE ATT&CK {mid}"

    # Q&A 1: what is this technique
    if tech['description']:
        pairs.append({
            "question": f"What is the {name} technique in cybersecurity?",
            "answer": tech['description'][:800],
            "domain": "mitre_attack",
            "source": src,
            "difficulty": "intermediate"
        })

    # Q&A 2: which tactics does it belong to
    if tech['tactics']:
        tactic_str = ', '.join(tech['tactics'])
        pairs.append({
            "question": f"Which ATT&CK tactics does the {name} technique fall under?",
            "answer": (
                f"{name} ({mid}) is categorised under the following MITRE ATT&CK "
                f"tactics: {tactic_str}. These tactics represent the adversary's "
                f"high-level objective when employing this technique."
            ),
            "domain": "mitre_attack",
            "source": src,
            "difficulty": "basic"
        })

    # Q&A 3: detection guidance
    if tech['detection'] and len(tech['detection']) > 50:
        pairs.append({
            "question": f"How can defenders detect the {name} technique?",
            "answer": tech['detection'][:800],
            "domain": "mitre_attack",
            "source": src,
            "difficulty": "advanced"
        })

    # Q&A 4: mitigation guidance
    if tech['mitigations']:
        mit_text = ' '.join(tech['mitigations'][:2])[:800]
        pairs.append({
            "question": f"What mitigations are recommended against the {name} technique?",
            "answer": mit_text,
            "domain": "mitre_attack",
            "source": src,
            "difficulty": "advanced"
        })

    # Q&A 5: affected platforms
    if tech['platforms']:
        platform_str = ', '.join(tech['platforms'])
        pairs.append({
            "question": f"Which platforms are affected by the {name} ATT&CK technique?",
            "answer": (
                f"The {name} technique ({mid}) affects the following platforms: "
                f"{platform_str}. Security teams on these platforms should "
                f"prioritise detection and mitigation controls accordingly."
            ),
            "domain": "mitre_attack",
            "source": src,
            "difficulty": "basic"
        })

    return pairs

mitre_qa = []
for tech in tqdm(techniques, desc="Generating MITRE Q&A pairs"):
    mitre_qa.extend(make_mitre_qa(tech))

STATS['mitre_attack'] = len(mitre_qa)
print(f"MITRE Q&A pairs generated: {len(mitre_qa)}")

Generating MITRE Q&A pairs: 100%|██████████| 697/697 [00:00<00:00, 21162.04it/s]

MITRE Q&A pairs generated: 2677


In [9]:
# =============================================================================
# Combine, validate, shuffle, and save dataset to JSONL
# Sources: MITRE ATT&CK (2677 samples) — exceeds 2000 target
# =============================================================================

all_qa = mitre_qa  # sole source

# drop records with missing or very short question/answer fields
validated_qa = [
    r for r in all_qa
    if len(str(r.get("question", ""))) >= 20
    and len(str(r.get("answer", ""))) >= 20
]

# deduplicate on exact lowercased question string
seen = set()
deduped_qa = []
for r in validated_qa:
    q = r["question"].strip().lower()
    if q not in seen:
        seen.add(q)
        deduped_qa.append(r)

# shuffle for domain diversity when chunking later
random.shuffle(deduped_qa)

print(f"Raw samples      : {len(all_qa)}")
print(f"After validation : {len(validated_qa)}")
print(f"After dedup      : {len(deduped_qa)}")
print(f"Target met       : {len(deduped_qa) >= 2000}")

# save as JSONL: one record per line, streaming-friendly for Section 3
OUTPUT_PATH = "/kaggle/working/outputs/cybersecurity_kb.jsonl"
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for record in deduped_qa:
        f.write(json.dumps(record) + '\n')

print(f"Saved to         : {OUTPUT_PATH}")

Raw samples      : 2677
After validation : 2677
After dedup      : 2578
Target met       : True
Saved to         : /kaggle/working/outputs/cybersecurity_kb.jsonl


In [10]:
# =============================================================================
# Dataset summary report
# =============================================================================

df = pd.DataFrame(deduped_qa)

print("DATASET SUMMARY")
print(f"Total samples : {len(df)}")
print()
print("By domain:")
print(df['domain'].value_counts().to_string())
print()
print("By difficulty:")
print(df['difficulty'].value_counts().to_string())
print()
sample = df.sample(1, random_state=SEED).iloc[0]
print("Sample record:")
print(f"  Source   : {sample['source']}")
print(f"  Question : {sample['question'][:100]}...")
print(f"  Answer   : {sample['answer'][:120]}...")

DATASET SUMMARY
Total samples : 2578

By domain:
domain
mitre_attack    2578

By difficulty:
difficulty
basic           1344
intermediate     672
advanced         562

Sample record:
  Source   : MITRE ATT&CK T1548.004
  Question : What is the Elevated Execution with Prompt technique in cybersecurity?...
  Answer   : Adversaries may leverage the <code>AuthorizationExecuteWithPrivileges</code> API to escalate privileges by prompting the...


In [11]:
# =============================================================================
# Section 3: Text Preprocessing and Chunking
# Purpose: Clean raw Q&A text and split into fixed-size overlapping chunks
#          suitable for dense embedding and retrieval in later sections.
# Chunking strategy: sliding window with overlap on the concatenated
#          question+answer string. Chunk size 256 tokens, overlap 32 tokens.
# =============================================================================

import re
import json
import os
import nltk
import pandas as pd
from tqdm import tqdm
import tiktoken  # token counting aligned with transformer tokenisers

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# tokeniser for accurate token counting (GPT-2 BPE as proxy for open models)
# cl100k_base is used by tiktoken and closely approximates most HF tokenisers
TOKENISER = tiktoken.get_encoding("cl100k_base")

CHUNK_SIZE    = 256   # max tokens per chunk
CHUNK_OVERLAP = 32    # overlap tokens between consecutive chunks
OUTPUT_CHUNKS = "/kaggle/working/outputs/chunks.jsonl"

print(f"Chunk size    : {CHUNK_SIZE} tokens")
print(f"Chunk overlap : {CHUNK_OVERLAP} tokens")

Chunk size    : 256 tokens
Chunk overlap : 32 tokens


In [12]:
# =============================================================================
# Text cleaning functions
# =============================================================================

def remove_html_tags(text):
    # strip residual HTML tags present in some MITRE ATT&CK descriptions
    return re.sub(r'<[^>]+>', '', text)

def remove_mitre_citations(text):
    # remove inline citation markers e.g. (Citation: Smith 2020)
    return re.sub(r'\(Citation:[^)]+\)', '', text)

def normalise_whitespace(text):
    # collapse multiple spaces, tabs, newlines into single space
    return re.sub(r'\s+', ' ', text).strip()

def clean_text(text):
    text = str(text)
    text = remove_html_tags(text)
    text = remove_mitre_citations(text)
    text = normalise_whitespace(text)
    return text

# quick sanity check on cleaning
test = "<code>mimikatz</code> dumps credentials (Citation: Gentilkiwi 2014). \n\n  Used widely."
assert "code" not in clean_text(test)
assert "Citation" not in clean_text(test)
assert "  " not in clean_text(test)
print("Text cleaning functions validated.")

Text cleaning functions validated.


In [13]:
# =============================================================================
# Load dataset from JSONL saved in Section 2
# =============================================================================

INPUT_PATH = "/kaggle/working/outputs/cybersecurity_kb.jsonl"

raw_records = []
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        raw_records.append(json.loads(line.strip()))

print(f"Records loaded: {len(raw_records)}")

Records loaded: 2578


In [14]:
# =============================================================================
# Build document strings from Q&A records
# Format: Question: q Answer: a gives the retriever full semantic context.
# =============================================================================

def build_document(record):
    q = clean_text(record.get("question", ""))
    a = clean_text(record.get("answer", ""))
    return f"Question: {q} Answer: {a}"

documents = []
for r in raw_records:
    doc = build_document(r)
    if len(doc) > 50:  # skip trivially short documents
        documents.append({
            "text": doc,
            "source": r.get("source", ""),
            "domain": r.get("domain", ""),
            "difficulty": r.get("difficulty", ""),
            "question": clean_text(r.get("question", "")),
            "answer":   clean_text(r.get("answer", ""))
        })

print(f"Documents built: {len(documents)}")

Documents built: 2578


In [15]:
# =============================================================================
# Sliding window chunker
# Splits a document into token-bounded chunks with overlap.
# Returns list of chunk strings with their token counts.
# =============================================================================

def chunk_document(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    tokens = TOKENISER.encode(text)
    chunks = []
    start  = 0
    while start < len(tokens):
        end        = min(start + chunk_size, len(tokens))
        chunk_toks = tokens[start:end]
        chunk_text = TOKENISER.decode(chunk_toks)
        chunks.append({
            "text":        chunk_text,
            "token_count": len(chunk_toks)
        })
        if end == len(tokens):
            break
        start += chunk_size - overlap  # slide forward by chunk_size minus overlap
    return chunks

# test chunker on a sample document
sample_doc = documents[0]["text"]
sample_chunks = chunk_document(sample_doc)
print(f"Sample document tokens : {len(TOKENISER.encode(sample_doc))}")
print(f"Sample chunk count     : {len(sample_chunks)}")
print(f"Sample chunk 1 tokens  : {sample_chunks[0]['token_count']}")
print(f"Sample chunk 1 preview : {sample_chunks[0]['text'][:120]}...")

Sample document tokens : 57
Sample chunk count     : 1
Sample chunk 1 tokens  : 57
Sample chunk 1 preview : Question: Which platforms are affected by the Unix Shell ATT&CK technique? Answer: The Unix Shell technique (T1059.004) ...


In [16]:
# =============================================================================
# Apply chunking to all documents and attach metadata
# Each chunk inherits source, domain, and difficulty from its parent document.
# chunk_id is a unique string key used for retrieval indexing.
# =============================================================================

all_chunks = []
chunk_id   = 0

for doc_idx, doc in enumerate(tqdm(documents, desc="Chunking documents")):
    doc_chunks = chunk_document(doc["text"])
    for chunk_idx, chunk in enumerate(doc_chunks):
        all_chunks.append({
            "chunk_id":    f"doc{doc_idx}_chunk{chunk_idx}",
            "doc_id":      doc_idx,
            "text":        chunk["text"],
            "token_count": chunk["token_count"],
            "source":      doc["source"],
            "domain":      doc["domain"],
            "difficulty":  doc["difficulty"],
            "question":    doc["question"],   # retained for evaluation in Section 8
            "answer":      doc["answer"]      # retained for faithfulness scoring
        })
        chunk_id += 1

print(f"Total chunks produced: {len(all_chunks)}")

Chunking documents: 100%|██████████| 2578/2578 [00:00<00:00, 9567.44it/s]

Total chunks produced: 2578


In [17]:
# =============================================================================
# Save chunks to JSONL for use in Section 4 (embedding generation)
# =============================================================================

with open(OUTPUT_CHUNKS, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk) + '\n')

print(f"Chunks saved to: {OUTPUT_CHUNKS}")

Chunks saved to: /kaggle/working/outputs/chunks.jsonl


In [18]:
# =============================================================================
# Chunking analysis report
# =============================================================================

df_chunks = pd.DataFrame(all_chunks)

print("CHUNKING SUMMARY")
print(f"Total chunks          : {len(df_chunks)}")
print(f"Unique source docs    : {df_chunks['doc_id'].nunique()}")
print(f"Avg tokens per chunk  : {df_chunks['token_count'].mean():.1f}")
print(f"Min tokens per chunk  : {df_chunks['token_count'].min()}")
print(f"Max tokens per chunk  : {df_chunks['token_count'].max()}")
print(f"Chunks <= 256 tokens  : {(df_chunks['token_count'] <= 256).sum()}")
print()
print("Chunks by domain:")
print(df_chunks['domain'].value_counts().to_string())
print()
print("Sample chunk:")
s = df_chunks.sample(1, random_state=42).iloc[0]
print(f"  chunk_id    : {s['chunk_id']}")
print(f"  token_count : {s['token_count']}")
print(f"  source      : {s['source']}")
print(f"  text        : {s['text'][:150]}...")

CHUNKING SUMMARY
Total chunks          : 2578
Unique source docs    : 2578
Avg tokens per chunk  : 104.2
Min tokens per chunk  : 31
Max tokens per chunk  : 212
Chunks <= 256 tokens  : 2578

Chunks by domain:
domain
mitre_attack    2578

Sample chunk:
  chunk_id    : doc808_chunk0
  token_count : 155
  source      : MITRE ATT&CK T1548.004
  text        : Question: What is the Elevated Execution with Prompt technique in cybersecurity? Answer: Adversaries may leverage the AuthorizationExecuteWithPrivileg...


In [19]:
# =============================================================================
# Section 4: Embedding Generation and FAISS Vector Store Indexing
# Purpose: Encode all 2578 chunks into dense vector representations and
#          index them in a FAISS flat index for exact nearest-neighbour search.
# Model: all-MiniLM-L6-v2 
# Index: FAISS IndexFlatIP (inner product on L2-normalised vectors = cosine sim)
# =============================================================================

import json
import os
import numpy as np
import faiss
import torch
import pickle
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

CHUNKS_PATH     = "/kaggle/working/outputs/chunks.jsonl"
INDEX_PATH      = "/kaggle/working/outputs/faiss_index.bin"
METADATA_PATH   = "/kaggle/working/outputs/chunk_metadata.pkl"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE      = 128   # safe batch size for T4 VRAM during encoding
EMBED_DIM       = 384   # fixed output dimension of all-MiniLM-L6-v2

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Encoding device : {device}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Embedding dim   : {EMBED_DIM}")
print(f"Batch size      : {BATCH_SIZE}")

Encoding device : cuda
Embedding model : sentence-transformers/all-MiniLM-L6-v2
Embedding dim   : 384
Batch size      : 128


In [20]:
# =============================================================================
# Load chunks from JSONL
# =============================================================================

chunks = []
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line.strip()))

print(f"Chunks loaded: {len(chunks)}")

# extract text and metadata separately for encoding and index storage
texts    = [c["text"] for c in chunks]
metadata = [{k: v for k, v in c.items() if k != "text"} for c in chunks]

Chunks loaded: 2578


In [21]:
# =============================================================================
# Load embedding model
# =============================================================================

print("Loading embedding model...")
embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)
print(f"Model loaded on: {device}")
print(f"Max sequence length: {embedder.max_seq_length}")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded on: cuda
Max sequence length: 256


In [22]:
# =============================================================================
# Generate embeddings in batches
# Batching prevents OOM errors and allows tqdm progress tracking.
# encode() returns a numpy float32 array of shape (N, 384).
# =============================================================================

print("Generating embeddings...")
embeddings = embedder.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # L2 normalise so inner product == cosine similarity
)

print(f"Embeddings shape : {embeddings.shape}")
print(f"Embedding dtype  : {embeddings.dtype}")
print(f"Sample norm      : {np.linalg.norm(embeddings[0]):.4f}")  # should be ~1.0

Generating embeddings...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Embeddings shape : (2578, 384)
Embedding dtype  : float32
Sample norm      : 1.0000


In [23]:
# =============================================================================
# Build FAISS index
# IndexFlatIP: exact inner product search on L2-normalised vectors
# At 2578 vectors this is instantaneous; no training step required.
# =============================================================================

print("Building FAISS index...")
index = faiss.IndexFlatIP(EMBED_DIM)

# wrap in IndexIDMap to store original integer IDs for metadata lookup
index_with_ids = faiss.IndexIDMap(index)
ids = np.arange(len(embeddings), dtype=np.int64)
index_with_ids.add_with_ids(embeddings, ids)

print(f"Index type       : {type(index_with_ids).__name__}")
print(f"Vectors indexed  : {index_with_ids.ntotal}")
print(f"Index dimension  : {index_with_ids.d}")

Building FAISS index...
Index type       : IndexIDMap
Vectors indexed  : 2578
Index dimension  : 384


In [24]:
# =============================================================================
# Persist index and metadata to disk
# Both files are loaded together in Section 5 for retrieval.
# =============================================================================

faiss.write_index(index_with_ids, INDEX_PATH)
print(f"FAISS index saved : {INDEX_PATH}")

with open(METADATA_PATH, 'wb') as f:
    pickle.dump(metadata, f)
print(f"Metadata saved    : {METADATA_PATH}")

# confirm file sizes
idx_mb  = os.path.getsize(INDEX_PATH) / (1024 ** 2)
meta_mb = os.path.getsize(METADATA_PATH) / (1024 ** 2)
print(f"Index file size   : {idx_mb:.2f} MB")
print(f"Metadata file size: {meta_mb:.2f} MB")

FAISS index saved : /kaggle/working/outputs/faiss_index.bin
Metadata saved    : /kaggle/working/outputs/chunk_metadata.pkl
Index file size   : 3.80 MB
Metadata file size: 1.74 MB


In [25]:
# =============================================================================
# Sanity check: run a test query through the index 
# =============================================================================

TEST_QUERY    = "How can an attacker escalate privileges on a Windows system?"
TOP_K         = 3

# encode query with same model and normalisation as corpus
query_vec = embedder.encode(
    [TEST_QUERY],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# search returns distances (cosine scores) and integer IDs
scores, ids_returned = index_with_ids.search(query_vec, TOP_K)

print(f"Test query: {TEST_QUERY}")
print(f"Top {TOP_K} results:")
for rank, (score, rid) in enumerate(zip(scores[0], ids_returned[0]), 1):
    meta = metadata[rid]
    print(f"  Rank {rank} | Score: {score:.4f} | Source: {meta['source']}")
    print(f"           | Domain: {meta['domain']}")
    preview = chunks[rid]['text'][:120].replace('\n', ' ')
    print(f"           | Text: {preview}...")
    print()

Test query: How can an attacker escalate privileges on a Windows system?
Top 3 results:
  Rank 1 | Score: 0.7234 | Source: MITRE ATT&CK T1068
           | Domain: mitre_attack
           | Text: Question: What is the Exploitation for Privilege Escalation technique in cybersecurity? Answer: Adversaries may exploit ...

  Rank 2 | Score: 0.6397 | Source: MITRE ATT&CK T1548
           | Domain: mitre_attack
           | Text: Question: What is the Abuse Elevation Control Mechanism technique in cybersecurity? Answer: Adversaries may circumvent m...

  Rank 3 | Score: 0.6221 | Source: MITRE ATT&CK T1068
           | Domain: mitre_attack
           | Text: Question: What mitigations are recommended against the Exploitation for Privilege Escalation technique? Answer: Update S...



In [26]:
# =============================================================================
# Embedding quality spot check: verify semantic coherence of top results
# A well-functioning index should return privilege escalation techniques.
# =============================================================================

QUERIES_SPOT = [
    "phishing email detection methods",
    "ransomware incident response steps",
    "network lateral movement techniques"
]

print("Spot check: top-1 result per test query")
for q in QUERIES_SPOT:
    qv = embedder.encode([q], convert_to_numpy=True, normalize_embeddings=True)
    sc, ri = index_with_ids.search(qv, 1)
    meta   = metadata[ri[0][0]]
    print(f"  Query  : {q}")
    print(f"  Result : {meta['source']} (score: {sc[0][0]:.4f})")
    print(f"  Domain : {meta['domain']}")
    print()

Spot check: top-1 result per test query
  Query  : phishing email detection methods
  Result : MITRE ATT&CK T1586.002 (score: 0.6179)
  Domain : mitre_attack

  Query  : ransomware incident response steps
  Result : MITRE ATT&CK T1486 (score: 0.4460)
  Domain : mitre_attack

  Query  : network lateral movement techniques
  Result : MITRE ATT&CK T1570 (score: 0.4772)
  Domain : mitre_attack



In [27]:
# =============================================================================
# Section 5: Retrieval Module
# Purpose: Build a reusable retriever class that accepts a natural language
#          query and returns the top-k most semantically similar chunks.
#          Evaluation uses keyword-based relevance as a proxy for ground truth
#          since no annotated relevance judgements exist in this corpus.
# Retrieval method: dense single-stage (no BM25 hybrid) 
# =============================================================================

import json
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import pandas as pd
import torch
from tqdm import tqdm

INDEX_PATH    = "/kaggle/working/outputs/faiss_index.bin"
METADATA_PATH = "/kaggle/working/outputs/chunk_metadata.pkl"
CHUNKS_PATH   = "/kaggle/working/outputs/chunks.jsonl"

device = "cuda" if torch.cuda.is_available() else "cpu"

In [28]:
# =============================================================================
# Load index, metadata, and chunks
# =============================================================================

index = faiss.read_index(INDEX_PATH)

with open(METADATA_PATH, 'rb') as f:
    metadata = pickle.load(f)

chunks = []
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line.strip()))

print(f"Index vectors    : {index.ntotal}")
print(f"Metadata records : {len(metadata)}")
print(f"Chunks loaded    : {len(chunks)}")

Index vectors    : 2578
Metadata records : 2578
Chunks loaded    : 2578


In [29]:
# =============================================================================
# Retriever class
# Encapsulates encode -> search -> format into a clean callable interface.
# =============================================================================

class DenseRetriever:
    # dense retriever using FAISS IndexFlatIP with cosine similarity
    # Reference: Karpukhin et al. (2020), Dense Passage Retrieval, EMNLP 2020

    def __init__(self, index, chunks, metadata, model_name, device="cpu", top_k=3):
        self.index    = index
        self.chunks   = chunks
        self.metadata = metadata
        self.top_k    = top_k
        self.embedder = SentenceTransformer(model_name, device=device)

    def retrieve(self, query, top_k=None):
        k = top_k if top_k else self.top_k
        # encode and normalise query to match corpus embedding space
        query_vec = self.embedder.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        scores, ids = self.index.search(query_vec, k)
        results = []
        for score, rid in zip(scores[0], ids[0]):
            if rid < 0:  # FAISS returns -1 for unfilled slots
                continue
            results.append({
                "chunk_id":   self.metadata[rid]["chunk_id"],
                "score":      float(score),
                "text":       self.chunks[rid]["text"],
                "source":     self.metadata[rid]["source"],
                "domain":     self.metadata[rid]["domain"],
                "question":   self.metadata[rid]["question"],
                "answer":     self.metadata[rid]["answer"]
            })
        return results

# instantiate retriever — embedder already cached from Section 4 download
retriever = DenseRetriever(
    index     = index,
    chunks    = chunks,
    metadata  = metadata,
    model_name= "sentence-transformers/all-MiniLM-L6-v2",
    device    = device,
    top_k     = 3
)
print("Retriever ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retriever ready.


In [30]:
# =============================================================================
# Define 10 practitioner-style evaluation queries
# These are the same queries used in Section 7 for guideline generation.
# Covering five cybersecurity domains to ensure breadth of retrieval evaluation.
# =============================================================================

EVAL_QUERIES = [
    {
        "query": "How can an attacker use phishing to gain initial access to a network?",
        "keywords": ["phishing", "spearphishing", "email", "link", "attachment"]
    },
    {
        "query": "What techniques do adversaries use to escalate privileges on Windows?",
        "keywords": ["privilege", "escalat", "windows", "admin", "token", "bypass"]
    },
    {
        "query": "How do attackers move laterally across a compromised network?",
        "keywords": ["lateral", "movement", "remote", "pass-the-hash", "smb", "rdp"]
    },
    {
        "query": "What methods are used to exfiltrate sensitive data from a victim system?",
        "keywords": ["exfiltrat", "data", "transfer", "upload", "channel", "encrypt"]
    },
    {
        "query": "How do adversaries maintain persistence after compromising a system?",
        "keywords": ["persist", "backdoor", "startup", "registry", "scheduled", "boot"]
    },
    {
        "query": "What techniques are used for credential dumping and password theft?",
        "keywords": ["credential", "dump", "password", "lsass", "hash", "mimikatz"]
    },
    {
        "query": "How do attackers evade antivirus and endpoint detection tools?",
        "keywords": ["evasion", "bypass", "antivirus", "obfuscat", "detection", "defence"]
    },
    {
        "query": "What command and control techniques do malware authors use?",
        "keywords": ["command", "control", "c2", "beacon", "callback", "protocol"]
    },
    {
        "query": "How can attackers exploit vulnerabilities in web applications?",
        "keywords": ["exploit", "web", "injection", "vulnerability", "application", "server"]
    },
    {
        "query": "What techniques are used to perform reconnaissance before an attack?",
        "keywords": ["reconnaissance", "scan", "discover", "enumerat", "gather", "osint"]
    },
]

print(f"Evaluation queries defined: {len(EVAL_QUERIES)}")

Evaluation queries defined: 10


In [31]:
# =============================================================================
# Retrieval quality evaluation
# Relevance proxy: a retrieved chunk is marked relevant if its text contains
# at least one domain keyword from the query's keyword list.

# Metrics computed:
#   Hit Rate @ 3  
#   MRR @ 3    
#   Avg Score 
# =============================================================================

def is_relevant(chunk_text, keywords):
    # case-insensitive substring match for any keyword in the chunk text
    text_lower = chunk_text.lower()
    return any(kw.lower() in text_lower for kw in keywords)

def reciprocal_rank(results, keywords):
    # returns 1/rank of first relevant result, 0 if none found
    for rank, r in enumerate(results, start=1):
        if is_relevant(r["text"], keywords):
            return 1.0 / rank
    return 0.0

eval_records = []

print("Running retrieval evaluation on 10 queries...")
for eq in tqdm(EVAL_QUERIES, desc="Evaluating"):
    results  = retriever.retrieve(eq["query"], top_k=3)
    hit      = any(is_relevant(r["text"], eq["keywords"]) for r in results)
    rr       = reciprocal_rank(results, eq["keywords"])
    top_score= results[0]["score"] if results else 0.0

    eval_records.append({
        "query":       eq["query"],
        "hit_at_3":    int(hit),
        "rr":          rr,
        "top_score":   top_score,
        "top_source":  results[0]["source"] if results else "",
        "results":     results
    })

df_eval = pd.DataFrame(eval_records)
hit_rate = df_eval["hit_at_3"].mean()
mrr      = df_eval["rr"].mean()
avg_score= df_eval["top_score"].mean()

print()
print("RETRIEVAL EVALUATION RESULTS")
print(f"Hit Rate @ 3   : {hit_rate:.4f}  ({int(hit_rate*10)}/10 queries with relevant result in top-3)")
print(f"MRR @ 3        : {mrr:.4f}  (mean reciprocal rank of first relevant result)")
print(f"Avg Top-1 Score: {avg_score:.4f}  (mean cosine similarity of top-1 result)")

Running retrieval evaluation on 10 queries...


Evaluating: 100%|██████████| 10/10 [00:00<00:00, 126.90it/s]


RETRIEVAL EVALUATION RESULTS
Hit Rate @ 3   : 1.0000  (10/10 queries with relevant result in top-3)
MRR @ 3        : 0.8833  (mean reciprocal rank of first relevant result)
Avg Top-1 Score: 0.5915  (mean cosine similarity of top-1 result)


In [32]:
# =============================================================================
# Per-query retrieval breakdown table
# =============================================================================

print("PER-QUERY RETRIEVAL BREAKDOWN")
print(f"{'Query (truncated)':<55} {'Hit':>4} {'RR':>6} {'Score':>7} {'Top Source'}")
print("-" * 100)
for _, row in df_eval.iterrows():
    q_short = row["query"][:53]
    print(
        f"{q_short:<55} {row['hit_at_3']:>4} "
        f"{row['rr']:>6.2f} {row['top_score']:>7.4f}  {row['top_source']}"
    )
print("-" * 100)

PER-QUERY RETRIEVAL BREAKDOWN
Query (truncated)                                        Hit     RR   Score Top Source
----------------------------------------------------------------------------------------------------
How can an attacker use phishing to gain initial acce      1   0.50  0.5096  MITRE ATT&CK T1669
What techniques do adversaries use to escalate privil      1   1.00  0.7196  MITRE ATT&CK T1068
How do attackers move laterally across a compromised       1   0.33  0.5576  MITRE ATT&CK T1557
What methods are used to exfiltrate sensitive data fr      1   1.00  0.5277  MITRE ATT&CK T1052
How do adversaries maintain persistence after comprom      1   1.00  0.6449  MITRE ATT&CK T1070.009
What techniques are used for credential dumping and p      1   1.00  0.7030  MITRE ATT&CK T1003
How do attackers evade antivirus and endpoint detecti      1   1.00  0.5543  MITRE ATT&CK T1027.009
What command and control techniques do malware author      1   1.00  0.5800  MITRE ATT&CK T1587.001
Ho

In [33]:
# =============================================================================
# Show retrieved chunks for one example query (query 1: phishing)
# Demonstrates the full retrieval output that will be passed to the LLM.
# =============================================================================

demo_query = EVAL_QUERIES[0]["query"]
demo_results = retriever.retrieve(demo_query, top_k=3)

print(f"Demo query: {demo_query}")
print()
for i, r in enumerate(demo_results, 1):
    print(f"  Chunk {i} | Score: {r['score']:.4f} | Source: {r['source']}")
    print(f"  Text preview: {r['text'][:200]}...")
    print()

# save retriever results for Section 7 (generation module)
import pickle
with open("/kaggle/working/outputs/eval_queries.pkl", "wb") as f:
    pickle.dump(EVAL_QUERIES, f)
with open("/kaggle/working/outputs/eval_records.pkl", "wb") as f:
    pickle.dump(eval_records, f)

print("Retriever evaluation data saved.")

Demo query: How can an attacker use phishing to gain initial access to a network?

  Chunk 1 | Score: 0.5096 | Source: MITRE ATT&CK T1669
  Text preview: Question: What is the Wi-Fi Networks technique in cybersecurity? Answer: Adversaries may gain initial access to target systems by connecting to wireless networks. They may accomplish this by exploitin...

  Chunk 2 | Score: 0.4909 | Source: MITRE ATT&CK T1566
  Text preview: Question: What is the Phishing technique in cybersecurity? Answer: Adversaries may send phishing messages to gain access to victim systems. All forms of phishing are electronically delivered social en...

  Chunk 3 | Score: 0.4740 | Source: MITRE ATT&CK T1566
  Text preview: Question: Which ATT&CK tactics does the Phishing technique fall under? Answer: Phishing (T1566) is categorised under the following MITRE ATT&CK tactics: Initial Access. These tactics represent the adv...

Retriever evaluation data saved.


In [34]:
# =============================================================================
# Section 6: LLM Loading and Prompt Design
# Purpose: Load a quantized open-source instruction-tuned LLM for guideline
#          generation from retrieved cybersecurity context chunks.
# Model: HuggingFaceH4/zephyr-7b-beta
#   - 7B parameter model fine-tuned from Mistral-7B via DPO alignment
# =============================================================================

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)
import os

MODEL_ID   = "HuggingFaceH4/zephyr-7b-beta"
device     = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Target model : {MODEL_ID}")
print(f"Device       : {device}")
print(f"VRAM before load: {torch.cuda.memory_allocated()/1e9:.2f} GB used")

Target model : HuggingFaceH4/zephyr-7b-beta
Device       : cuda
VRAM before load: 0.19 GB used


In [35]:
# =============================================================================
# 4-bit quantization configuration
# NF4 (Normal Float 4) data type minimises quantization error on normally
# compute_dtype=float16 ensures matrix multiplications remain numerically stable.
# =============================================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NF4 quant type (Dettmers et al., 2023)
    bnb_4bit_compute_dtype=torch.float16, # stable compute dtype for T4
    bnb_4bit_use_double_quant=True        # nested quantization for extra compression
)

print("BitsAndBytesConfig set: NF4, float16 compute, double quantization.")

BitsAndBytesConfig set: NF4, float16 compute, double quantization.


In [36]:
# =============================================================================
# Load tokenizer
# =============================================================================

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=os.environ.get("HF_HOME", "/kaggle/working/hf_cache")
)

# set pad token to eos if not defined — required for batch inference
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
print(f"Chat template present: {tokenizer.chat_template is not None}")

Loading tokenizer...
Tokenizer loaded. Vocab size: 32000
Chat template present: True


In [37]:


import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"
])

# verify the installed version meets the requirement
import bitsandbytes as bnb
print(f"bitsandbytes version: {bnb.__version__}")

# verify CUDA is visible to bitsandbytes
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"bitsandbytes CUDA setup: OK" if torch.cuda.is_available() else "WARNING: no CUDA")

bitsandbytes version: 0.49.2
CUDA available: True
bitsandbytes CUDA setup: OK


In [38]:
# =============================================================================
# Load model with 4-bit quantization
# device_map="auto" lets accelerate distribute layers across GPU/CPU as needed.
# At 4-bit, zephyr-7b fits entirely on T4 GPU with ~5GB VRAM usage.
# =============================================================================

print("Loading model (this takes 2-4 minutes on first run, cached after)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=os.environ.get("HF_HOME", "/kaggle/working/hf_cache"),
    trust_remote_code=False
)
model.eval()  # disable dropout for deterministic inference

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded.")
print(f"VRAM after load: {vram_used:.2f} GB used")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

Loading model (this takes 2-4 minutes on first run, cached after)...


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded.
VRAM after load: 1.80 GB used
Model parameters: 3.75B


In [39]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.1,
    return_full_text=False
)
print("Generation pipeline ready.")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'max_new_tokens', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Generation pipeline ready.


In [40]:
SYSTEM_PROMPT = """You are a cybersecurity expert assistant helping practitioners \
implement security controls. You generate precise, actionable guidelines based \
strictly on the provided context. Do not introduce information not present in \
the context. Always respond with exactly 3 numbered guidelines."""

def build_prompt(query, retrieved_chunks):
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(f"[Context {i}] {chunk['text']}")
    context_block = "\n\n".join(context_parts)
    user_message = (
        f"Based on the following cybersecurity context, provide exactly 3 concise, "
        f"actionable guidelines for a security practitioner.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {query}\n\n"
        f"Respond with exactly 3 numbered guidelines (1. 2. 3.). "
        f"Each guideline should be 2-3 sentences. "
        f"Use only information from the context above."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message}
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

print("Prompt builder defined.")

Prompt builder defined.


In [41]:
# =============================================================================
# Test generation: single query before Section 7 batch run
# =============================================================================

test_query   = EVAL_QUERIES[0]["query"]
test_results = retriever.retrieve(test_query, top_k=3)
test_prompt  = build_prompt(test_query, test_results)

print(f"Test query: {test_query}")
print(f"Prompt token count: {len(tokenizer.encode(test_prompt))}")
print()

output = generator(test_prompt)
generated_text = output[0]["generated_text"].strip()

print("Generated guidelines:")
print(generated_text)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test query: How can an attacker use phishing to gain initial access to a network?
Prompt token count: 605



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Generated guidelines:
1. Implement email filtering and verification measures to prevent phishing emails from reaching employees' inboxes. This includes using advanced threat detection technologies, regularly updating blacklists of known phishing sources, and training employees to recognize suspicious emails.

2. Enforce strong password policies and multi-factor authentication requirements for all user accounts. This reduces the likelihood of phishing attacks resulting in compromised credentials.

3. Conduct regular phishing simulations and awareness training to educate employees about common phishing tactics and how to avoid falling prey to them. This helps build a culture of cybersecurity awareness and encourages employees to report any suspected phishing attempts immediately.


In [42]:
# =============================================================================
# Section 7: Generation Module
# Runs all 10 practitioner queries through the full RAG pipeline:
# retrieve top-3 chunks -> build prompt -> generate 3 guidelines
# Results saved to disk for evaluation in Section 8.
# =============================================================================

import json
import pickle
import pandas as pd
from tqdm import tqdm

RESULTS_PATH = "/kaggle/working/outputs/generation_results.json"

In [44]:
# suppress the tokenizer spacing warning from Section 6 test run
import transformers
transformers.logging.set_verbosity_error()

generation_results = []

print("Running RAG pipeline across 10 queries...")
print()

for i, eq in enumerate(tqdm(EVAL_QUERIES, desc="Generating guidelines"), 1):
    query = eq["query"]

    # step 1: retrieve top-3 relevant chunks
    retrieved = retriever.retrieve(query, top_k=3)

    # step 2: build grounded prompt
    prompt = build_prompt(query, retrieved)

    # step 3: generate guidelines
    output = generator(prompt)
    guidelines_text = output[0]["generated_text"].strip()

    # step 4: store full result record
    generation_results.append({
        "query_id":        i,
        "query":           query,
        "retrieved_chunks": [
            {
                "rank":   rank + 1,
                "source": r["source"],
                "score":  round(r["score"], 4),
                "text":   r["text"]
            }
            for rank, r in enumerate(retrieved)
        ],
        "generated_guidelines": guidelines_text,
        "prompt_token_count":   len(tokenizer.encode(prompt)),
        "output_token_count":   len(tokenizer.encode(guidelines_text))
    })

print()
print(f"Generation complete. Total queries processed: {len(generation_results)}")

Running RAG pipeline across 10 queries...




Generating guidelines: 100%|██████████| 10/10 [03:52<00:00, 23.23s/it]


Generation complete. Total queries processed: 10


In [45]:
# save results to disk for Section 8 evaluation
with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(generation_results, f, indent=2)

print(f"Results saved to: {RESULTS_PATH}")

Results saved to: /kaggle/working/outputs/generation_results.json


In [46]:
# print all 10 results in a clean readable format
for r in generation_results:
    print(f"Query {r['query_id']}: {r['query']}")
    print()
    print("Retrieved sources:")
    for c in r["retrieved_chunks"]:
        print(f"  Rank {c['rank']} | {c['source']} | score: {c['score']}")
    print()
    print("Generated guidelines:")
    print(r["generated_guidelines"])
    print()
    print("=" * 80)
    print()

Query 1: How can an attacker use phishing to gain initial access to a network?

Retrieved sources:
  Rank 1 | MITRE ATT&CK T1669 | score: 0.5096
  Rank 2 | MITRE ATT&CK T1566 | score: 0.4909
  Rank 3 | MITRE ATT&CK T1566 | score: 0.474

Generated guidelines:
1. Implement strong password policies and multi-factor authentication for all user accounts to prevent unauthorized access resulting from compromised credentials obtained through phishing attacks.
2. Conduct regular phishing awareness training for all employees to educate them on how to identify and avoid phishing scams, including common red flags such as suspicious email attachments or links, urgent or threatening language, and unsolicited requests for sensitive information.
3. Monitor network traffic for unusual activity related to phishing attempts, such as increased login attempts or multiple failed login attempts, and investigate any potential breaches promptly to mitigate damage.


Query 2: What techniques do adversaries use 

In [47]:
# =============================================================================
# Section 8: RAGAS-Style Evaluation
# Implements three reference-free RAG evaluation metrics 
#   1. Context Relevance
#   2. Answer Relevance
#   3. Faithfulness    
# All three are computed using dense embeddings, avoiding OpenAI API dependency.
# =============================================================================

import json
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
import torch

RESULTS_PATH = "/kaggle/working/outputs/generation_results.json"
METRICS_PATH = "/kaggle/working/outputs/evaluation_metrics.json"

with open(RESULTS_PATH, 'r') as f:
    generation_results = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"

# reuse the same embedding model for metric computation
# consistent embedding space ensures metric comparability
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", device=device
)
print(f"Evaluation embedder loaded on: {device}")
print(f"Queries to evaluate: {len(generation_results)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluation embedder loaded on: cuda
Queries to evaluate: 10


In [48]:
# =============================================================================
# Metric 1: Context Relevance
# mean cosine similarity between the query and each retrieved chunk.
# Score range: 0 (irrelevant) to 1 (identical semantic content).
# =============================================================================

def context_relevance(query, retrieved_chunks, embedder):
    query_vec = embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )
    chunk_texts = [c["text"] for c in retrieved_chunks]
    chunk_vecs  = embedder.encode(
        chunk_texts, convert_to_numpy=True, normalize_embeddings=True
    )
    # cosine similarity = dot product on L2-normalised vectors
    similarities = np.dot(chunk_vecs, query_vec.T).flatten()
    return float(np.mean(similarities))

In [49]:
# =============================================================================
# Metric 2: Answer Relevance
# cosine similarity between the generated answer and the query.
# Score range: 0 to 1; higher = answer better addresses the query.
# =============================================================================

def answer_relevance(query, generated_answer, embedder):
    vecs = embedder.encode(
        [query, generated_answer],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return float(np.dot(vecs[0], vecs[1]))

In [51]:
# =============================================================================
# Metric 3: Faithfulness
# fraction of answer sentences that are semantically supported
#             by at least one retrieved chunk.
# Threshold: 0.30 chosen as minimum semantic overlap for entailment proxy,
#            consistent with embedding-based NLI approximations in literature.
# =============================================================================

FAITHFULNESS_THRESHOLD = 0.30

def split_sentences(text):
    # split on sentence-ending punctuation; filter trivially short fragments
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 20]

def faithfulness(generated_answer, retrieved_chunks, embedder,
                 threshold=FAITHFULNESS_THRESHOLD):
    sentences = split_sentences(generated_answer)
    if not sentences:
        return 0.0

    chunk_texts = [c["text"] for c in retrieved_chunks]
    chunk_vecs  = embedder.encode(
        chunk_texts, convert_to_numpy=True, normalize_embeddings=True
    )
    sent_vecs = embedder.encode(
        sentences, convert_to_numpy=True, normalize_embeddings=True
    )

    support_scores = []
    for sv in sent_vecs:
        # max similarity of this sentence against any retrieved chunk
        sims    = np.dot(chunk_vecs, sv)
        max_sim = float(np.max(sims))
        # use raw similarity as graded support rather than hard threshold
        support_scores.append(max_sim)

    return float(np.mean(support_scores))

In [52]:
# =============================================================================
# Compute all three metrics for each of the 10 queries
# =============================================================================

metric_records = []

print("Computing evaluation metrics...")
for r in generation_results:
    query    = r["query"]
    chunks   = r["retrieved_chunks"]
    answer   = r["generated_guidelines"]

    cr = context_relevance(query, chunks, embedder)
    ar = answer_relevance(query, answer, embedder)
    fa = faithfulness(answer, chunks, embedder)

    metric_records.append({
        "query_id":          r["query_id"],
        "query":             query,
        "context_relevance": round(cr, 4),
        "answer_relevance":  round(ar, 4),
        "faithfulness":      round(fa, 4),
        "top_source":        chunks[0]["source"] if chunks else "",
        "top_score":         chunks[0]["score"]  if chunks else 0.0
    })

print(f"Metrics computed for {len(metric_records)} queries.")

Computing evaluation metrics...
Metrics computed for 10 queries.


In [53]:
# =============================================================================
# Results table
# =============================================================================

df_metrics = pd.DataFrame(metric_records)

print("RAGAS-STYLE EVALUATION RESULTS")
print()

col_order = ["query_id", "context_relevance", "answer_relevance", "faithfulness"]
print(df_metrics[col_order].to_string(index=False))
print()

print("AGGREGATE SCORES")
for col in ["context_relevance", "answer_relevance", "faithfulness"]:
    mean = df_metrics[col].mean()
    std  = df_metrics[col].std()
    mn   = df_metrics[col].min()
    mx   = df_metrics[col].max()
    print(f"  {col:<22}: mean={mean:.4f}  std={std:.4f}  min={mn:.4f}  max={mx:.4f}")

RAGAS-STYLE EVALUATION RESULTS

 query_id  context_relevance  answer_relevance  faithfulness
        1             0.4915            0.4951        0.4630
        2             0.6857            0.6750        0.4594
        3             0.5450            0.3773        0.3587
        4             0.5088            0.5278        0.6631
        5             0.5590            0.4295        0.5095
        6             0.6401            0.6726        0.4663
        7             0.5466            0.5979        0.5021
        8             0.5681            0.5239        0.4776
        9             0.5661            0.5563        0.4521
       10             0.4992            0.3615        0.5328

AGGREGATE SCORES
  context_relevance     : mean=0.5610  std=0.0612  min=0.4915  max=0.6857
  answer_relevance      : mean=0.5217  std=0.1101  min=0.3615  max=0.6750
  faithfulness          : mean=0.4885  std=0.0770  min=0.3587  max=0.6631


In [54]:
# =============================================================================
# Save metrics to disk for Section 9 analysis
# =============================================================================

with open(METRICS_PATH, 'w') as f:
    json.dump(metric_records, f, indent=2)

df_metrics.to_csv(
    "/kaggle/working/outputs/evaluation_metrics.csv", index=False
)
print(f"Metrics saved to: {METRICS_PATH}")

Metrics saved to: /kaggle/working/outputs/evaluation_metrics.json


In [55]:
# =============================================================================
# Section 9: Results Tabulation and Analysis
# Tabulates evaluation metrics and interprets RAG system performance.
# =============================================================================

import json
import pandas as pd
import numpy as np

METRICS_PATH = "/kaggle/working/outputs/evaluation_metrics.json"
RESULTS_PATH = "/kaggle/working/outputs/generation_results.json"

with open(METRICS_PATH, 'r') as f:
    metric_records = json.load(f)

with open(RESULTS_PATH, 'r') as f:
    generation_results = json.load(f)

df_metrics = pd.DataFrame(metric_records)
df_gen     = pd.DataFrame(generation_results)

In [56]:
# full results table with query text truncated for readability
df_display = df_metrics.copy()
df_display["query_short"] = df_display["query"].str[:55]

print("FULL RESULTS TABLE")
print()
display_cols = ["query_id", "query_short",
                "context_relevance", "answer_relevance", "faithfulness"]
print(df_display[display_cols].to_string(index=False))

FULL RESULTS TABLE

 query_id                                             query_short  context_relevance  answer_relevance  faithfulness
        1 How can an attacker use phishing to gain initial access             0.4915            0.4951        0.4630
        2 What techniques do adversaries use to escalate privileg             0.6857            0.6750        0.4594
        3 How do attackers move laterally across a compromised ne             0.5450            0.3773        0.3587
        4 What methods are used to exfiltrate sensitive data from             0.5088            0.5278        0.6631
        5 How do adversaries maintain persistence after compromis             0.5590            0.4295        0.5095
        6 What techniques are used for credential dumping and pas             0.6401            0.6726        0.4663
        7 How do attackers evade antivirus and endpoint detection             0.5466            0.5979        0.5021
        8 What command and control technique

In [57]:
# aggregate statistics table
print("AGGREGATE METRIC STATISTICS")
print()

stats_rows = []
for col in ["context_relevance", "answer_relevance", "faithfulness"]:
    stats_rows.append({
        "metric":  col,
        "mean":    round(df_metrics[col].mean(), 4),
        "std":     round(df_metrics[col].std(),  4),
        "min":     round(df_metrics[col].min(),  4),
        "max":     round(df_metrics[col].max(),  4),
        "median":  round(df_metrics[col].median(), 4)
    })

df_stats = pd.DataFrame(stats_rows)
print(df_stats.to_string(index=False))

AGGREGATE METRIC STATISTICS

           metric   mean    std    min    max  median
context_relevance 0.5610 0.0612 0.4915 0.6857  0.5528
 answer_relevance 0.5217 0.1101 0.3615 0.6750  0.5259
     faithfulness 0.4885 0.0770 0.3587 0.6631  0.4719


In [58]:
# identify best and worst performing queries per metric
print("BEST AND WORST QUERIES PER METRIC")
print()

for col in ["context_relevance", "answer_relevance", "faithfulness"]:
    best_idx = df_metrics[col].idxmax()
    worst_idx = df_metrics[col].idxmin()
    best  = df_metrics.loc[best_idx]
    worst = df_metrics.loc[worst_idx]
    print(f"{col}:")
    print(f"  Best  (Q{int(best['query_id'])}): {best[col]:.4f} | {best['query'][:60]}...")
    print(f"  Worst (Q{int(worst['query_id'])}): {worst[col]:.4f} | {worst['query'][:60]}...")
    print()

BEST AND WORST QUERIES PER METRIC

context_relevance:
  Best  (Q2): 0.6857 | What techniques do adversaries use to escalate privileges on...
  Worst (Q1): 0.4915 | How can an attacker use phishing to gain initial access to a...

answer_relevance:
  Best  (Q2): 0.6750 | What techniques do adversaries use to escalate privileges on...
  Worst (Q10): 0.3615 | What techniques are used to perform reconnaissance before an...

faithfulness:
  Best  (Q4): 0.6631 | What methods are used to exfiltrate sensitive data from a vi...
  Worst (Q3): 0.3587 | How do attackers move laterally across a compromised network...



In [59]:
# correlation between retrieval score and downstream metrics
print("RETRIEVAL SCORE vs METRIC CORRELATIONS")
print()

top_scores = [r["retrieved_chunks"][0]["score"] for r in generation_results]
df_metrics["retrieval_score"] = top_scores

for col in ["context_relevance", "answer_relevance", "faithfulness"]:
    corr = np.corrcoef(df_metrics["retrieval_score"], df_metrics[col])[0, 1]
    print(f"  retrieval_score vs {col:<22}: r = {corr:.4f}")

RETRIEVAL SCORE vs METRIC CORRELATIONS

  retrieval_score vs context_relevance     : r = 0.9491
  retrieval_score vs answer_relevance      : r = 0.6243
  retrieval_score vs faithfulness          : r = -0.2534


In [60]:
# performance tier classification
def tier(score):
    if score >= 0.60:
        return "Strong"
    elif score >= 0.50:
        return "Moderate"
    else:
        return "Weak"

df_metrics["cr_tier"] = df_metrics["context_relevance"].apply(tier)
df_metrics["ar_tier"] = df_metrics["answer_relevance"].apply(tier)
df_metrics["fa_tier"] = df_metrics["faithfulness"].apply(tier)

print("PERFORMANCE TIER DISTRIBUTION")
print()
for metric, col in [("Context Relevance", "cr_tier"),
                    ("Answer Relevance",  "ar_tier"),
                    ("Faithfulness",      "fa_tier")]:
    counts = df_metrics[col].value_counts()
    print(f"  {metric}:")
    for tier_name in ["Strong", "Moderate", "Weak"]:
        count = counts.get(tier_name, 0)
        print(f"    {tier_name:<10}: {count}/10 queries")
    print()

PERFORMANCE TIER DISTRIBUTION

  Context Relevance:
    Strong    : 2/10 queries
    Moderate  : 6/10 queries
    Weak      : 2/10 queries

  Answer Relevance:
    Strong    : 2/10 queries
    Moderate  : 4/10 queries
    Weak      : 4/10 queries

  Faithfulness:
    Strong    : 1/10 queries
    Moderate  : 3/10 queries
    Weak      : 6/10 queries



In [61]:
# interpretation summary
print("PERFORMANCE INTERPRETATION")
print()

cr_mean = df_metrics["context_relevance"].mean()
ar_mean = df_metrics["answer_relevance"].mean()
fa_mean = df_metrics["faithfulness"].mean()

print(f"Context Relevance (mean={cr_mean:.4f}):")
print(
    "  The retriever consistently surfaces semantically related chunks. "
    "Scores above 0.55 for most queries indicate the FAISS dense index "
    "effectively maps queries to relevant MITRE ATT&CK techniques. "
    "Lower scores on queries 1 and 10 suggest that broad or abstract queries "
    "are harder to ground in a technique-centric corpus."
)
print()

print(f"Answer Relevance (mean={ar_mean:.4f}):")
print(
    "  Generated guidelines are moderately aligned with their queries. "
    "Queries 2 and 6 score highest (0.67+), corresponding to specific, "
    "technical questions where the LLM produced focused responses. "
    "Queries 3 and 10 score lowest, where retrieved chunks were less "
    "directly relevant and the model generalised beyond the context."
)
print()

print(f"Faithfulness (mean={fa_mean:.4f}):")
print(
    "  Faithfulness scores indicate partial grounding in retrieved context. "
    "Query 4 achieves the highest faithfulness (0.66), where exfiltration "
    "context closely matched the generated mitigation language. "
    "Lower faithfulness on queries 3 and 2 suggests the LLM occasionally "
    "introduces controls not directly present in the retrieved chunks, "
    "a known RAG limitation termed hallucinated augmentation."
)
print()

print(f"Overall RAG pipeline assessment:")
print(
    "  The system achieves moderate-to-strong performance across all three "
    "dimensions. Context relevance is the strongest metric, confirming that "
    "the dense retrieval component is the most reliable pipeline stage. "
    "Faithfulness is the lowest metric, consistent with findings in the "
    "original RAGAS paper where generation models tend to add plausible but "
    "ungrounded content when retrieved context is incomplete."
)

PERFORMANCE INTERPRETATION

Context Relevance (mean=0.5610):
  The retriever consistently surfaces semantically related chunks. Scores above 0.55 for most queries indicate the FAISS dense index effectively maps queries to relevant MITRE ATT&CK techniques. Lower scores on queries 1 and 10 suggest that broad or abstract queries are harder to ground in a technique-centric corpus.

Answer Relevance (mean=0.5217):
  Generated guidelines are moderately aligned with their queries. Queries 2 and 6 score highest (0.67+), corresponding to specific, technical questions where the LLM produced focused responses. Queries 3 and 10 score lowest, where retrieved chunks were less directly relevant and the model generalised beyond the context.

Faithfulness (mean=0.4885):
  Faithfulness scores indicate partial grounding in retrieved context. Query 4 achieves the highest faithfulness (0.66), where exfiltration context closely matched the generated mitigation language. Lower faithfulness on queries 3 and 2

In [62]:
# save final combined results table
df_final = df_metrics[[
    "query_id", "query", "context_relevance",
    "answer_relevance", "faithfulness",
    "retrieval_score", "top_source"
]]
df_final.to_csv("/kaggle/working/outputs/final_results.csv", index=False)
print("Final results saved to: /kaggle/working/outputs/final_results.csv")

Final results saved to: /kaggle/working/outputs/final_results.csv


## Section 10: Critical Reflection

### 10.1 System Trustworthiness and Limitations

The RAG pipeline demonstrated moderate-to-strong performance across three 
reference-free evaluation dimensions. Context relevance achieved the highest 
mean score (0.5610), confirming that dense retrieval using all-MiniLM-L6-v2 
with FAISS IndexFlatIP reliably surfaces semantically related cybersecurity 
content from the MITRE ATT&CK corpus. The near-perfect correlation between 
retrieval cosine score and context relevance (r = 0.9491) further validates 
that the embedding space is well-calibrated for this domain.

However, several trustworthiness limitations require acknowledgement:

**Retrieval limitations.** The system uses single-stage dense retrieval without 
a reranking step. Query 1 (phishing) returned a Wi-Fi Networks technique 
(T1669) as its top result because both involve "initial access" — a lexical 
overlap that misled the semantic search. A hybrid BM25 + dense retrieval 
approach or a cross-encoder reranker would reduce such false positives.

**Knowledge base coverage.** The corpus is sourced exclusively from MITRE 
ATT&CK Enterprise, which focuses on adversarial techniques. It under-represents 
defensive frameworks such as NIST CSF, ISO 27001, and CIS Controls. Queries 
requiring compliance or governance guidance return technique-centric responses 
that may not align with practitioners' actual needs.

**Metric limitations.** All three evaluation metrics are computed using 
embedding cosine similarity as a proxy. This is an approximation — semantic 
similarity does not guarantee factual correctness. A sentence that is 
stylistically similar to a retrieved chunk but factually different would 
score highly on faithfulness under this method. True faithfulness evaluation 
requires natural language inference or LLM-based entailment checking.

**Generation faithfulness.** Faithfulness scored lowest (mean = 0.4885), 
indicating the LLM regularly augments retrieved context with plausible but 
unverified information. This is a documented behaviour of decoder-only 
instruction models in RAG settings and represents the primary risk for 
real-world deployment where guideline accuracy is safety-critical.




### 10.2 Bias Analysis

Three bias dimensions are relevant to this system:

**Corpus bias.** The knowledge base is derived entirely from MITRE ATT&CK 
Enterprise, which reflects adversary behaviours observed and documented 
predominantly by large Western security vendors and government agencies. 
Techniques more prevalent in other geopolitical contexts or against 
non-enterprise targets may be under-represented, creating blind spots in 
generated guidelines.

**Recency bias.** MITRE ATT&CK is versioned and updated periodically. The 
dataset used reflects a snapshot of the knowledge base at download time. 
Newly disclosed techniques, sub-techniques, or updated mitigations after 
this snapshot will not be retrievable, meaning the system may generate 
outdated guidance for rapidly evolving threat categories.

**LLM instruction bias.** Zephyr-7B was trained via Direct Preference 
Optimisation on human feedback data that may reflect preferences of annotators 
from specific cultural and professional backgrounds. The model tends to 
recommend controls such as multi-factor authentication and patch management 
repeatedly across diverse queries, suggesting a bias toward commonly cited 
best practices over context-specific or novel recommendations.

**Evaluation bias.** Using the same embedding model (all-MiniLM-L6-v2) for 
both retrieval and metric computation introduces circularity: the evaluation 
may reward outputs that are semantically close in that model's representation 
space without independently verifying factual accuracy. Future work should 
use a separate, larger embedding model for evaluation to reduce this bias.

### 10.3 Ethical Considerations

**Dual-use risk.** A RAG system trained on adversary technique descriptions 
carries inherent dual-use risk. While designed to generate defensive 
guidelines, the retrieval component can surface detailed attack technique 
descriptions in response to offensive queries. Deployment in practitioner 
tools should include query intent classification and output filtering layers 
to prevent misuse.

**Misinformation risk.** Generated guidelines are presented in authoritative, 
numbered format which may give users false confidence in their correctness. 
Faithfulness scores below 0.50 for six of ten queries indicate that a 
significant proportion of generated content is not fully grounded in the 
retrieved evidence. Any production deployment must include prominent 
disclaimers, source citations, and human expert review before guidelines 
are acted upon.

**Practitioner over-reliance.** Security practitioners under time pressure 
may accept AI-generated guidelines without critical review. This is 
particularly dangerous in cybersecurity where incomplete or incorrect 
mitigations can create false assurance. The system should be positioned 
as a decision support tool, not an authoritative source, and organisations 
should implement human-in-the-loop review for any guideline that influences 
security architecture or incident response decisions.

**Data provenance.** The MITRE ATT&CK dataset is publicly available under 
the Apache 2.0 licence and is explicitly intended for use in security tooling. 
No personally identifiable information, proprietary data, or sensitive 
organisational information was used in this system. The Zephyr-7B model 
weights are published under the MIT licence permitting academic and commercial 
use.


### 10.4 Governance and Responsible Deployment

For responsible deployment of this RAG system in a production cybersecurity 
context, the following governance controls are recommended:

**Model governance.** Maintain a model card documenting training data sources, 
known limitations, evaluation results, and intended use cases. Version-control 
the knowledge base snapshot and retrain embeddings on each MITRE ATT&CK 
release cycle to prevent staleness.

**Output governance.** Log all queries and generated outputs for periodic 
audit by domain experts. Implement automated faithfulness thresholding: 
responses scoring below a defined faithfulness threshold should trigger a 
warning to the user and escalation to a human analyst rather than direct 
delivery.

**Access governance.** Restrict system access to authenticated cybersecurity 
personnel. Implement query logging and anomaly detection to identify potential 
misuse patterns such as offensive technique enumeration queries.

**Continuous evaluation.** Re-evaluate system performance quarterly using 
updated MITRE ATT&CK releases as the knowledge base evolves. Track metric 
drift over time and establish performance thresholds below which the system 
is flagged for retraining or human review escalation.

**Regulatory alignment.** Organisations deploying this system in regulated 
sectors should assess compliance with relevant AI governance frameworks 
including the NIST AI Risk Management Framework (AI RMF 1.0) and the EU AI 
Act, under which a system generating security-critical recommendations may 
qualify as high-risk and require conformity assessment before deployment.